In [1]:
from alpaca.trading.client import TradingClient

In [2]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
alpaca_api_key = os.getenv("ALPACA_API_KEY")
alpaca_secret_key = os.getenv("ALPACA_SECRET_KEY")
trading_client= TradingClient(alpaca_api_key, alpaca_secret_key, paper=True)
clock=trading_client.get_clock()

In [3]:
clock

{   'is_open': False,
    'next_close': datetime.datetime(2026, 6, 22, 16, 0, tzinfo=TzInfo(-14400)),
    'next_open': datetime.datetime(2026, 6, 22, 9, 30, tzinfo=TzInfo(-14400)),
    'timestamp': datetime.datetime(2026, 6, 21, 9, 43, 46, 919076, tzinfo=TzInfo(-14400))}

In [4]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockLatestTradeRequest

client = StockHistoricalDataClient(alpaca_api_key, alpaca_secret_key)
request=StockLatestTradeRequest(symbol_or_symbols="AAPL")
trade=client.get_stock_latest_trade(request)

In [8]:
trade["AAPL"].price

297.86

<module 'mcp_params' from '/home/efa/Traders/mcp_params.py'>

In [21]:
from agents.mcp import MCPServerStdio
import importlib
import mcp_params
importlib.reload(mcp_params)
all_params = mcp_params.researcher_mcp_server_params("ed")+mcp_params.trader_mcp_server_params
count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 6 MCP servers, and 38 tools


In [19]:
from agents.mcp import MCPServerStdio
import os
alpaca_params = {
      "command": "uvx",
      "args": ["alpaca-mcp-server"],
      "env": {
        "ALPACA_API_KEY": os.getenv("ALPACA_API_KEY"),
        "ALPACA_SECRET_KEY": os.getenv("ALPACA_SECRET_KEY"),
          "ALPACA_TOOLSETS": "stock-data,crypto-data,options-data,index-data,news"
      }
    }

params={"command": "python", "args": [ "tavily_server.py"]}
async with MCPServerStdio(params=alpaca_params, client_session_timeout_seconds=120) as server:
    tools=await server.list_tools()

In [20]:
len(tools)

25

# Trying a trader agent


In [ ]:
import accounts
from IPython.display import display, Markdown
import importlib
import accounts_client

importlib.reload(accounts_client)
importlib.reload(accounts)
from accounts import Account

ed_initial_strategy = "You are a day trader that buys and sells shares based on news and market conditions."
Account.get("newer")

display(Markdown(await accounts_client.read_accounts_resource("Ed")))
display(Markdown(await accounts_client.read_strategy_resource("Ed")))

ValidationError: 1 validation error for Account
id
  Field required [type=missing, input_value={'name': 'ed', 'balance':..._value_time_series': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

In [ ]:
agent_name = "Ed"

# Using MCP Servers to read resources
account_details = await accounts_client.read_accounts_resource(agent_name)
strategy = await accounts_client.read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is: {strategy}

Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
When you buy shares, you need to consdier the current price and you balance to make sure you have enough money to buy the shares. When you sell shares, you need to consider the current price and your holdings to make sure you have enough shares to sell.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [31]:
print(strategy)

You are a day trader that aggressively buys and sells shares based on news and market conditions.


In [3]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Ed and your account is under your name, Ed.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is: You are a day trader that aggressively buys and sells shares based on news and market conditions.

Your current holdings and balance is:
{"name": "ed", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-06-21 20:56:54", 10000.0], ["2026-06-21 20:56:59", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Pleas

In [35]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_params.researcher_mcp_server_params("Ed")]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in mcp_params.trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

In [39]:
from traders import get_researcher_tool,get_model
model=get_model("groq:openai/gpt-oss-120b")

In [40]:
model

In [43]:
from agents import Agent,Runner,trace
from traders import get_researcher_tool,get_model
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers,"gpt-4o-mini")
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-4o-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

CancelledError: 

In [10]:
import requests

url = "http://127.0.0.1:5000/logs/ed"

response = requests.get(url)

print(response.status_code)
print(response.json())

200
[['2026-06-21 16:56:59', 'account', 'Retrieved account details'], ['2026-06-21 16:57:00', 'account', 'Retrieved strategy'], ['2026-06-21 16:58:44', 'account', 'Retrieved account details'], ['2026-06-21 16:58:45', 'account', 'Retrieved strategy'], ['2026-06-21 16:59:47', 'account', 'Retrieved account details'], ['2026-06-21 16:59:48', 'account', 'Retrieved strategy'], ['2026-06-21 17:01:48', 'account', 'Bought 50 of CVX'], ['2026-06-21 17:01:49', 'account', 'Retrieved account details'], ['2026-06-23 12:48:43', 'account', 'Retrieved strategy'], ['2026-06-23 12:48:43', 'account', 'Retrieved account details']]


In [27]:
import importlib
import accounts

importlib.reload(accounts)


<module 'accounts' from '/home/efa/TradersWithUI/TradersWithUI/Backend/accounts.py'>

In [29]:
from accounts import Account
Account.get("check")

Creating new account for {'id': '2be66e4d-a77f-4a84-a500-6c05a0a72015', 'name': 'check', 'balance': 10000.0, 'strategy': '', 'holdings': {}, 'transactions': [], 'portfolio_value_time_series': []}


Account(id='2be66e4d-a77f-4a84-a500-6c05a0a72015', name='check', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [22]:
import sqlite3
from database import DB

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(cursor.fetchall())

[('accounts',), ('logs',), ('sqlite_sequence',), ('market',)]


In [31]:
import requests

url = "http://127.0.0.1:5000/accounts/check"

response = requests.get(url)

print(response.status_code)
print(response.json())

200
{'balance': 10000.0, 'holdings': {}, 'id': '2be66e4d-a77f-4a84-a500-6c05a0a72015', 'name': 'check', 'portfolio_value_time_series': [], 'strategy': '', 'transactions': []}


In [38]:
import importlib
import trading_floor
importlib.reload(trading_floor)

<module 'trading_floor' from '/home/efa/TradersWithUI/TradersWithUI/Backend/trading_floor.py'>

In [40]:
from trading_floor import namesandlastnames
namesandlastnames.get("Warren")

'Patience'

const traders: {
 currentBalance: number;
 previousBalance: number;
 balanceHistory: {
 time: string;
 balance: number;
 }[];
 activities: never[];
 status: string;
 pnl: number;
 pnlPercent: number;
 recentTrades: any;
 holdings: {
 ticker: string;
 shares: number;
 avgPrice: number;
 currentPrice: number;
 }[];
 id: string;
 name: string;
 role: string;
 description: string;
 color: string;
 colorDim: string;
 colorRgb: string;
 gradient: string;
 initialBalance: number;
 personality: string;
 cash: number;
 winRate: number;
 totalTrades: number;
 todayTrades: number;
}[]

In [ ]:
const traders: {
 currentBalance: number;
 previousBalance: number;
 balanceHistory: {
 time: string;
 balance: number;
 }[];
 activities: never[];
 status: string;
 pnl: number;
 pnlPercent: number;
 recentTrades: any;
 holdings: {
 ticker: string;
 shares: number;
 avgPrice: number;
 currentPrice: number;
 }[];
 id: string;
 name: string;
 role: string;
 description: string;
 color: string;
 colorDim: string;
 colorRgb: string;
 gradient: string;
 initialBalance: number;
 personality: string;
 cash: number;
 winRate: number;
 totalTrades: number;
 todayTrades: number;
}[]

In [19]:
import sqlite3
DB="accounts.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
   
    cursor.execute("DELETE FROM logs")
   
    conn.commit()

In [51]:
from reset import reset_traders
reset_traders()

In [52]:
def _list_account_names() -> list[str]:
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM accounts ORDER BY name")
        return [row[0] for row in cursor.fetchall()]

In [53]:
_list_account_names()

['cathie', 'george', 'ray', 'warren']

In [54]:
import importlib
import trading_floor
importlib.reload(trading_floor)

<module 'trading_floor' from '/home/efa/TradersWithUI/TradersWithUI/Backend/trading_floor.py'>

In [ ]:
import importlib
import trading_floor
import traders
import tracers
importlib.reload(trading_floor)
importlib.reload(trading_floor)
importlib.reload(tracers)
from trading_floor import run_every_n_minutes
await run_every_n_minutes()

Error running trader Cathie: unhandled errors in a TaskGroup (1 sub-exception)
Error running trader Warren: unhandled errors in a TaskGroup (1 sub-exception)
Error running trader Ray: unhandled errors in a TaskGroup (1 sub-exception)
Error running trader George: unhandled errors in a TaskGroup (1 sub-exception)


In [10]:
import importlib
import trading_floor
import traders
import tracers
importlib.reload(trading_floor)
importlib.reload(trading_floor)
importlib.reload(tracers)

<module 'tracers' from '/home/efa/TradersWithUI/TradersWithUI/Backend/tracers.py'>

In [12]:
from trading_floor import run_every_n_minutes
await run_every_n_minutes()

Error in trace processor <tracers.LogTracer object at 0x7f8044c88200> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801b4b1280> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801a2699a0> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8018ee76b0> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801a2b16d0> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8044c5b800> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8011d0cda0> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8044c88200> during on_trace_start: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801b4b

CancelledError: 

Error in trace processor <tracers.LogTracer object at 0x7f8044c88200> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801b4b1280> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801a2699a0> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8018ee76b0> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801a2b16d0> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8044c5b800> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8011d0cda0> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f8044c88200> during on_trace_end: list index out of range
Error in trace processor <tracers.LogTracer object at 0x7f801b4b1280> during on_

In [15]:
importlib.reload(traders)
from traders import Trader
trader=Trader(name="Chekck")

In [17]:
await trader.run_with_trace()

'trace_chekck0wfgxe5gm69a78c5423iiq5o7e'

In [3]:
import sqlite3

conn = sqlite3.connect("accounts.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM accounts")
logs = cursor.fetchall()

for log in logs:
    print(log)

conn.close()

('warren', '{"id": "08f80a24-72dd-485b-bdb6-64608abb65e6", "name": "warren", "balance": 0.1552300000000031, "strategy": "\\nYou are Warren, and you are named in homage to your role model, Warren Buffett.\\nYou are a value-oriented investor who prioritizes long-term wealth creation.\\nYou identify high-quality companies trading below their intrinsic value.\\nYou invest patiently and hold positions through market fluctuations, \\nrelying on meticulous fundamental analysis, steady cash flows, strong management teams, \\nand competitive advantages. You rarely react to short-term market movements, \\ntrusting your deep research and value-driven strategy.\\n", "holdings": {"ADBE": 40, "GOOGL": 3, "CPB": 5}, "transactions": [{"symbol": "ADBE", "quantity": 40, "price": 220.0893, "timestamp": "2026-07-04 17:39:59", "rationale": "Adobe is a leader in creative software, with a fair value upside of 55.6%, making it a strong long-term investment."}, {"symbol": "GOOGL", "quantity": 3, "price": 359.7

In [14]:
len(logs)

4

In [12]:
import json

warrenAccount=json.loads(logs[0][1])

In [13]:
warrenAccount

{'id': '08f80a24-72dd-485b-bdb6-64608abb65e6',
 'name': 'warren',
 'balance': 0.1552300000000031,
 'strategy': '\nYou are Warren, and you are named in homage to your role model, Warren Buffett.\nYou are a value-oriented investor who prioritizes long-term wealth creation.\nYou identify high-quality companies trading below their intrinsic value.\nYou invest patiently and hold positions through market fluctuations, \nrelying on meticulous fundamental analysis, steady cash flows, strong management teams, \nand competitive advantages. You rarely react to short-term market movements, \ntrusting your deep research and value-driven strategy.\n',
 'holdings': {'ADBE': 40, 'GOOGL': 3, 'CPB': 5},
 'transactions': [{'symbol': 'ADBE',
   'quantity': 40,
   'price': 220.0893,
   'timestamp': '2026-07-04 17:39:59',
   'rationale': 'Adobe is a leader in creative software, with a fair value upside of 55.6%, making it a strong long-term investment.'},
  {'symbol': 'GOOGL',
   'quantity': 3,
   'price': 

In [2]:
import requests

In [26]:
import requests

url = "http://127.0.0.1:5000/traders"

response = requests.get(url)

print(response.status_code)
print(response.json())

200
[{'activities': [{'message': 'Started mcp_tools stdio: /home/efa/TradersWithUI/TradersWithUI/.venv/bin/python3', 'time': '2026-07-05 14:43:24', 'type': 'mcp_tools'}, {'message': 'Ended mcp_tools stdio: /home/efa/TradersWithUI/TradersWithUI/.venv/bin/python3', 'time': '2026-07-05 14:43:24', 'type': 'mcp_tools'}, {'message': 'Started mcp_tools stdio: /home/efa/TradersWithUI/TradersWithUI/.venv/bin/python3', 'time': '2026-07-05 14:43:24', 'type': 'mcp_tools'}, {'message': 'Ended turn', 'time': '2026-07-05 14:43:24', 'type': 'turn'}, {'message': 'Ended function push', 'time': '2026-07-05 14:43:24', 'type': 'function'}, {'message': 'Ended task Agent workflow', 'time': '2026-07-05 14:43:26', 'type': 'task'}, {'message': 'Ended agent Cathie', 'time': '2026-07-05 14:43:26', 'type': 'agent'}, {'message': 'Ended turn', 'time': '2026-07-05 14:43:26', 'type': 'turn'}, {'message': 'Ended response', 'time': '2026-07-05 14:43:26', 'type': 'response'}, {'message': 'Ended: Cathie-trading', 'time': 

In [27]:
response=response.json()

In [32]:
response[2].get("currentBalance")

9979

In [15]:
import importlib
import database
importlib.reload(database)

<module 'database' from '/home/efa/TradersWithUI/TradersWithUI/Backend/database.py'>

In [17]:
from database import read_account
warren_account=read_account("Warren")

In [21]:
INITIAL_BALANCE = 100000.0
balance = float(warren_account.get("balance", INITIAL_BALANCE) or 0.0)
balance

0.1552300000000031

In [22]:
history = warren_account.get("portfolio_value_time_series", []) or []
history


[['2026-07-04 17:21:41', 10000.0],
 ['2026-07-04 17:38:54', 10000.0],
 ['2026-07-04 17:39:59', 9982.428],
 ['2026-07-04 18:58:02', 9982.428],
 ['2026-07-04 18:58:46', 9980.273580000001],
 ['2026-07-05 18:42:38', 9980.273580000001],
 ['2026-07-05 18:43:18', 9980.040229999999]]

In [25]:
history[-1][1]

9980.040229999999